# Linear SVC Hyperparameter Tuning - full spectrum

**Purpose:** Train and evaluate linear support vector classifiers on the **full 1003-wavenumber spectrum** with nested cross-validation. These results are the 'SVC full' reference lines in Fig. 3 of the manuscript (Sections 3.2 and 4.2).

The classifier is scikit-learn's `SVC(kernel="linear")`, with the penalty `C` tuned over 8 log-spaced values from 10$^{-3}$ to 10$^{3}$ (Table 1).

**Nested CV:** the outer loop leaves one biological replicate out (3 folds); within each outer fold `GridSearchCV` (default 5-fold, accuracy scoring) selects `C` on the training spectra, the best model is refit on all training spectra and evaluated on the held-out replicate. Three classifiers are trained: species, benzoate condition (`metabolites` in the code) and culture age. `random_state = 42`.

**Scaling:** within each outer fold a `StandardScaler` is fit on the training spectra only (each wavenumber z-scored) and applied to the held-out replicate, so no information from the test replicate enters the scaling. The fitted scaler is not saved to the pickle.

**Output:** `Performance Dictionaries/Performance_SVC_nCV.pkl`, same structure as `Performance_ANN_nCV.pkl`: `{ "CV k": {task: [X_train, y_train, X_test, y_test, y_predict, GridSearchCV, best model, confusion matrix, report]} }`.

In [1]:
# Imports (several are not used in this notebook, e.g. plotly, KernelPCA, SVR, RandomizedSearchCV, 3D plotting).
import pandas as pd
pd.DataFrame.iteritems = pd.DataFrame.items
import numpy as np
import os
import re
import pickle

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits import mplot3d
import matplotlib.ticker as mticker
import matplotlib.transforms as transforms
import matplotlib.patches as patches
import matplotlib.cm as cm

import plotly.express as px
import scipy

from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA, KernelPCA

from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier
from sklearn import metrics
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# Load the preprocessed dataset produced by Data_Preprocessing_Pipeline.ipynb
filename = "Preprocessed_Raman_Spectra_2025.pkl"
with open(filename, "rb") as f:
    loaded_data = pickle.load(f)
    
print(loaded_data['Spectra'].shape)

(360, 1003)


In [3]:
# Show the preprocessing settings stored with the data
loaded_data["Preprocess conditions"]

{'order': 'SI --> CS --> BSR --> SG --> ED',
 'CS': [1, 0.01],
 'BSR': [100, 1],
 'SG': [15, 2],
 'ED': [400, 1800]}

In [4]:
# Unpack the dictionary into separate arrays (spectra, wavenumbers, labels, replicate IDs)
wavenumber = loaded_data["wavenumber"]
spectral = loaded_data["Spectra"]

bacteria_species = loaded_data["species label"]
culture_age = loaded_data['culture age']
cul_cond = loaded_data["culture condition"]

biological_rep = loaded_data["biological replicate"]
technical_rep = loaded_data["technical replicate"]

In [5]:
# Translate the condition codes into readable labels: A = 0 mM benzoate, C = 20 mM benzoate
condition_dict = {"A" : "0 mM benzoate", "C" : "20 mM benzoate"}
culture_condition = []
for i in range(len(cul_cond)):
    culture_condition.append(condition_dict[cul_cond[i]])
culture_condition = np.array(culture_condition)

# Biological Replicate Seperation

In [6]:
# Build the label matrix y_data_full (one row per spectrum, three columns).
# Labels are integer-encoded in sorted order of the unique values:
#   col 0 = species:   CBG (C. glutamicum) = 0, PP (P. putida) = 1
#   col 1 = condition: 0 mM benzoate = 0, 20 mM benzoate = 1  (called 'metabolites' in the code below)
#   col 2 = culture age: 5 h = 0, 24 h = 1, 48 h = 2
# X_data_full is the matrix of preprocessed spectra (360 x 1003).
label_species = np.unique(bacteria_species)
label_condition = np.unique(culture_condition)
label_age = np.unique(culture_age)

species_dict = {}
for i in range(len(label_species)):
    species_dict[label_species[i]] = i
    
condition_dict = {}
for i in range(len(label_condition)):
    condition_dict[label_condition[i]] = i
    
age_dict = {}
for i in range(len(label_age)):
    age_dict[label_age[i]] = i
    
y_output = []
for i in range(len(bacteria_species)):
    temp = [species_dict[bacteria_species[i]],
           condition_dict[culture_condition[i]],
            age_dict[culture_age[i]]
           ]
    y_output.append(np.array(temp))

y_data_full = np.array(y_output)
X_data_full = spectral

print(X_data_full.shape, y_data_full.shape)

(360, 1003) (360, 3)


In [7]:
# Evaluate one outer fold: for each of the three classification tasks tune a linear SVC by grid search on the training spectra, then predict the held-out spectra.
# Inputs are the z-scored full spectra (the scaling is done in the outer-fold loop below; no PCA in this full-spectrum version).
# Returns {task: [X_train, y_train, X_test, y_test, y_predict, GridSearchCV object, best model, confusion matrix, classification report]}.
def nCV_eval(train_X, test_X, train_y, test_y):
    # Names of the three classification tasks. 'metabolites' = benzoate condition (0 vs 20 mM).
    names = ["species", "metabolites", "age"]
    overall_performance = {}
    rs = 42

    for j in range(len(names)):

        # Training / test inputs: all 1003 wavenumbers; labels: column j of the label matrix
        X_data_train = train_X
        y_data_train = train_y[:,j]

        X_data_test = test_X
        y_data_test = test_y[:,j]

        print(names[j], X_data_train.shape, y_data_train.shape)
        
        # Hyperparameter grid (Table 1): linear kernel, C = 8 log-spaced values from 1e-3 to 1e3 (C = inverse regularisation strength)
        param_grid = { "kernel": ["linear"],
                      "C" : np.logspace(-3, 3, 8)}

        # Support vector classifier with a fixed seed
        SVC_model = SVC(random_state = 42)

        # Inner cross-validation: GridSearchCV (5-fold by default, scored by accuracy); n_jobs = -1 uses all CPU cores
        SVC_grid = GridSearchCV(SVC_model, 
                                 param_grid, 
                                 n_jobs = -1, 
                                 verbose = 0, 
                                 return_train_score = True)

        SVC_grid.fit(X_data_train, y_data_train)

        # Best C found by the grid search (refit on all training spectra), evaluated on the held-out replicate
        SVC_best_model = SVC_grid.best_estimator_
        y_predict = SVC_best_model.predict(X_data_test)

        # Test-set classification report and confusion matrix
        report = classification_report(y_data_test, y_predict, output_dict = True)
        con_mat = confusion_matrix(y_data_test, y_predict)

        # Store inputs, predictions, grid search, best model and metrics
        overall_performance[names[j]] = [X_data_train, y_data_train, X_data_test, y_data_test, 
                                         y_predict, SVC_grid, SVC_best_model, con_mat, report]

    return overall_performance

In [8]:
# Outer loop = leave-one-biological-replicate-out (3-fold CV). Each fold trains on two biological replicates and tests on the held-out one,
# so spectra from the same culture never appear in both training and test sets (manuscript Section 3.2).
# Results for each fold are collected in NCV_performance['CV <replicate>'].
import time

# Biological replicate IDs (1, 2, 3); each takes a turn as the test set
uni_br = np.unique(biological_rep)
NCV_performance = {}

for i in range(len(uni_br)):
    strcv = "CV " + str(uni_br[i])
    
    print(strcv)
    
    # Boolean masks: spectra from the other replicates (train) and from the held-out replicate (test)
    ind_train = biological_rep != uni_br[i]
    ind_test = biological_rep == uni_br[i]
    
    train_X_full_us = X_data_full[ind_train,:]
    test_X_full_us = X_data_full[ind_test,:]
    
    # Z-score the spectra: the scaler is fit on the TRAINING spectra of this outer fold only (zero mean, unit variance per wavenumber)
    # and then applied unchanged to the held-out replicate, so nothing from the test replicate leaks into the scaling. (train_X_full_us / test_X_full_us = unscaled)
    SS_overall = StandardScaler()
    
    train_X_full = SS_overall.fit_transform(train_X_full_us)
    test_X_full = SS_overall.transform(test_X_full_us)
        
    train_y_data = y_data_full[ind_train,:]
    test_y_data = y_data_full[ind_test,:]
    
        
    # Tune and evaluate the three SVCs for this fold on the scaled spectra
    temp = nCV_eval(train_X_full,
                    test_X_full, 
                    train_y_data, 
                    test_y_data)


    # Print the test accuracy of each classifier for this fold
    for n in list(temp.keys()):
        print(n + " accuracy: ", round(temp[n][-1]["accuracy"], 2))

    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
        
    NCV_performance[strcv] = temp
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

CV 1
species (240, 1003) (240,)
metabolites (240, 1003) (240,)
age (240, 1003) (240,)
species accuracy:  0.94
metabolites accuracy:  0.76
age accuracy:  0.85
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
CV 2
species (240, 1003) (240,)
metabolites (240, 1003) (240,)
age (240, 1003) (240,)
species accuracy:  0.88
metabolites accuracy:  0.72
age accuracy:  0.85
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
CV 3
species (240, 1003) (240,)
metabolites (240, 1003) (240,)
age (240, 1003) (240,)
species accuracy:  0.85
metabolites accuracy:  0.73
age accuracy:  0.8
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [9]:
# Save the nested-CV results (input for ML_Performance_Comparison.ipynb)
import pickle

my_dictionary = NCV_performance
 
# Open a file in write binary mode ('wb')
with open('Performance Dictionaries\\Performance_SVC_nCV.pkl', 'wb') as file:
    # Serialize the dictionary and write it to the file
    pickle.dump(my_dictionary, file)